# 01 — Veri Hazırlama (İP-2)

Amaç: yönetmelik korpusundan chunk listesi üretmek ve soru–kanıt eşlemelerinin yazılacağı şemayı hazırlamak.

Önce `00_kurulum_ve_kontrol.ipynb` çalışmış olmalı (repo `/content/RAG-proje` altında).

In [ ]:
import sys
sys.path.insert(0, '/content/RAG-proje')
%cd /content/RAG-proje

from src.config import load_config
from src.data import korpus_yukle, yonetmelik_parcala, chunklari_kaydet

cfg = load_config()
metin = korpus_yukle(cfg['veri']['korpus'])
chunklar = yonetmelik_parcala(
    metin,
    boyut=cfg['parcalama']['boyut_karakter'],
    ortusme=cfg['parcalama']['ortusme_karakter'],
)
chunklari_kaydet(chunklar, 'data/processed/chunklar.jsonl')
print(f'{len(chunklar)} chunk yazıldı: data/processed/chunklar.jsonl')
for c in chunklar[:3]:
    print(c.chunk_id, '|', c.metin[:90], '...')

## Soru yazma görevi (Araştırmacı 1)

Her madde için 2–4 doğal soru yazın. Şema: `data/README.md`. Kritik kurallar:

- Her sorunun `kanit_chunk_id` alanı, cevabı içeren chunk'ı göstermeli.
- Aynı sorunun paraphrase varyantları aynı `varyant_grup` kimliğini alır ve **aynı bölmede** kalır.
- Soru tipleri: `olgusal`, `paraphrase`, `bicimbilimsel`, `cok-maddeli`.
- Hedef: 300–500 özgün soru. Örnek kayıtlar: `data/processed/sorular-ornek.jsonl`.

In [ ]:
# Şema doğrulama: soru dosyası yazıldıktan sonra bu hücreyi çalıştırın.
from src.data import jsonl_oku

yol = 'data/processed/sorular-ornek.jsonl'  # gerçeği: sorular.jsonl
sorular = jsonl_oku(yol)
gerekli = {'soru_id', 'soru', 'tip', 'kanit_chunk_id', 'varyant_grup', 'bolme'}
chunk_idler = {c.chunk_id for c in chunklar}

sorun = []
for s in sorular:
    eksik = gerekli - set(s)
    if eksik:
        sorun.append((s.get('soru_id'), f'eksik alan: {eksik}'))
    elif s['kanit_chunk_id'] not in chunk_idler:
        sorun.append((s['soru_id'], 'kanıt chunk bulunamadı'))

# varyant grubu bölme sızıntısı kontrolü
grup_bolme = {}
for s in sorular:
    grup_bolme.setdefault(s['varyant_grup'], set()).add(s['bolme'])
sizinti = [g for g, b in grup_bolme.items() if len(b) > 1]

print(f'{len(sorular)} soru denetlendi. Sorun: {sorun or "yok"}')
print(f'Bölme sızıntısı olan gruplar: {sizinti or "yok"}')
assert not sorun and not sizinti, 'Önce veri sorunlarını düzeltin.'
print('Veri hazır. Sıradaki adım: 02_retrieval_deneyleri.ipynb')